In [6]:
import pandas as pd
from collections import Counter

In [7]:
# FUNGSI DAN PROSEDUR #
import re

# konstanta untuk ekstraksi fitur dari teks pesan
BATAS_KATA_PENDEK = 5      # pesan <= 5 kata dianggap pendek
KATA_SPAM = ("free", "win", "call", "text", "claim", "prize", "urgent", "reward")
SIMBOL_SPAM = "!$#@%"

# ekstrak fitur kategorikal dari satu pesan teks
def ekstrak_fitur(pesan):
    p = str(pesan).lower()
    # pakai word-boundary supaya "free" di "freedom" tidak ikut terdeteksi
    ada_kata_spam = any(re.search(r"\b" + re.escape(k) + r"\b", p) for k in KATA_SPAM)
    return {
        "panjang": "pendek" if len(p.split()) <= BATAS_KATA_PENDEK else "panjang",
        "angka": "ada" if any(c.isdigit() for c in p) else "tidak",
        "simbol": "ada" if any(c in SIMBOL_SPAM for c in p) else "tidak",
        "kata_spam": "ada" if ada_kata_spam else "tidak",
    }

# Buat Model Naive Bayes
def naiveBayes(x, y):
    model = {}
    output = Counter(y)
    N = len(y)
    probabilitas_fitur = {}  #Likelihood

    model["prior"] = {
        k: output[k]/N
        for k in output
    }

    for k in output:
        probabilitas_fitur[k] = {}
        x_k = [x[i] for i in range(len(x)) if y[i] == k]  #ambil semua data dengan output k

        for f in range(len(x[0])):
            probabilitas_fitur[k][f] = {}  #kelas K, fitur f
            fitur_f = [xf[f] for xf in x_k]  #ambil semua fitur f dari kelas k
            kategori_count = Counter(fitur_f)  #hitung jumlah kemunculan setiap kategori
            n_kategori = len(fitur_f)

            for v in kategori_count:
                probabilitas_fitur[k][f][v] = kategori_count[v] / n_kategori  #kelas K, fitur f, probabilitas v

    model["likelihood"] = probabilitas_fitur
    model["jumlah_kelas"] = output
    model["jumlah_data"] = N

    return model

In [8]:
# Prediksi
def prediksi_naiveBayes(model, x):
    probabilitas = {}

    # Menghitung prior x likelihood untuk setiap kelas
    for k in model["prior"]:
        probabilitas_x = model["prior"][k]

        for f in range(len(x)):
            likelihood = model["likelihood"][k][f].get(x[f], 1e-6)
            probabilitas_x *= likelihood

        probabilitas[k] = probabilitas_x

    # Normalisasi agar menjadi probabilitas posterior
    total = sum(probabilitas.values())

    if total == 0:
        posterior = {k: 0 for k in probabilitas}
    else:
        posterior = {
            k: probabilitas[k] / total
            for k in probabilitas
        }

    # Kelas dengan probabilitas terbesar
    kelas_prediksi = max(posterior, key=posterior.get)

    return kelas_prediksi, probabilitas, posterior

# akurasi
def akurasi(y_asli, y_pred):
    benar = sum(1 for a, b in zip(y_asli, y_pred) if a == b)
    return benar / len(y_asli)

# RMSE (butuh label berbentuk angka 0/1)
def rmse(y_asli, y_pred):
    se = sum((a - b) ** 2 for a, b in zip(y_asli, y_pred))
    return (se / len(y_asli)) ** 0.5


In [9]:
# inisialisasi data
import kagglehub
from pathlib import Path

# download dataset dari kaggle (sekali download, run berikutnya pakai cache)
path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

# cari file csv di folder hasil download
data = next(Path(path).glob("*.csv"), None)
if data is None:
    raise FileNotFoundError("file csv tidak ditemukan di hasil download kaggle")

# membaca data: v1 = label (ham/spam), v2 = isi pesan
df = pd.read_csv(data, encoding="latin-1")[["v1", "v2"]]
df.columns = ["label", "pesan"]

# memastikan data tidak memiliki nilai kosong
df = df.dropna().copy()

# ekstraksi fitur kategorikal dari isi pesan (fungsi ekstrak_fitur ada di cell fungsi)
# supaya tetap cocok dengan model naive bayes kategorikal yang sudah ada
fitur_df = df["pesan"].apply(ekstrak_fitur).apply(pd.Series)
df = pd.concat([df, fitur_df], axis=1)

fitur = ["panjang", "angka", "simbol", "kata_spam"]
target = "label"

# memisahkan fitur dan target
x = df[fitur].values.tolist()
y = df[target].values.tolist()

print(len(x), len(y))
print(x[:3], y[:3])

5572 5572
[['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'tidak', 'ada']] ['ham', 'ham', 'spam']


In [10]:
# MAIN PROGRAM #

model_naiveBayes = naiveBayes(x, y)

# data uji: pesan pendek, ada angka, tanpa simbol, tanpa kata spam -> mirip ham
x_uji = ["pendek", "ada", "tidak", "tidak"]
print("\nData uji:")
for i in range(len(fitur)):
    print(fitur[i], "=", x_uji[i])

prediksi, probabilitas, posterior = prediksi_naiveBayes(
    model_naiveBayes,
    x_uji
)

print("\nProbabilitas:")
for kelas, p in posterior.items():
    print(f"{kelas}: {p:.4%}")

print("\nkelas hasil prediksi:", prediksi)


Data uji:
panjang = pendek
angka = ada
simbol = tidak
kata_spam = tidak

Probabilitas:
ham: 99.0651%
spam: 0.9349%

kelas hasil prediksi: ham


In [11]:
# langkah pengujian: split data latih/uji 80/20, hitung akurasi dan RMSE
import random
random.seed(42)

# ubah label jadi angka untuk RMSE: ham=0, spam=1 (urut alfabet)
label_angka = {k: i for i, k in enumerate(sorted(set(y)))}
y_num = [label_angka[v] for v in y]

idx = list(range(len(x)))
random.shuffle(idx)
batas = int(0.8 * len(x))
train_idx, test_idx = idx[:batas], idx[batas:]

x_latih = [x[i] for i in train_idx]
y_latih = [y[i] for i in train_idx]
x_uji = [x[i] for i in test_idx]
y_uji = [y[i] for i in test_idx]
y_uji_num = [y_num[i] for i in test_idx]

# bangun model dari data latih
model_latih = naiveBayes(x_latih, y_latih)

# prediksi data uji
y_pred = []
y_pred_num = []
for xi in x_uji:
    kelas, prob, posterior = prediksi_naiveBayes(model_latih, xi)
    y_pred.append(kelas)
    y_pred_num.append(label_angka[kelas])

# hitung akurasi dan RMSE pada data uji
acc = akurasi(y_uji, y_pred)
err = rmse(y_uji_num, y_pred_num)

print("data latih:", len(x_latih), "| data uji:", len(x_uji))
print("akurasi data uji: %.4f (%.2f%%)" % (acc, acc * 100))
print("RMSE data uji: %.4f" % err)

data latih: 4457 | data uji: 1115
akurasi data uji: 0.9552 (95.52%)
RMSE data uji: 0.2118


In [12]:
# eksperimen perhitungan manual dengan 20 data pertama
x20 = x[:20]
y20 = y[:20]

print("20 data:", x20)
print("label  :", y20)
print()

# model dari 20 data
model20 = naiveBayes(x20, y20)
print("prior:", {k: round(v, 4) for k, v in model20["prior"].items()})
print("jumlah kelas:", dict(model20["jumlah_kelas"]))
print()

# likelihood tiap fitur per kelas
for k in model20["prior"]:
    print("likelihood kelas", k, ":")
    for f in range(len(fitur)):
        print("  ", fitur[f], "->", {a: round(b, 4) for a, b in model20["likelihood"][k][f].items()})
print()

# pengujian ulang 20 data (resubstitution)
y_pred20 = []
posterior20 = []
for xi in x20:
    kelas, prob, post = prediksi_naiveBayes(model20, xi)
    y_pred20.append(kelas)
    posterior20.append(post)

acc20 = akurasi(y20, y_pred20)
label_angka20 = {k: i for i, k in enumerate(sorted(set(y20)))}
y20_num = [label_angka20[v] for v in y20]
y_pred20_num = [label_angka20[v] for v in y_pred20]
err20 = rmse(y20_num, y_pred20_num)

print("prediksi:", y_pred20)
print("akurasi 20 data: %.4f (%.2f%%)" % (acc20, acc20 * 100))
print("RMSE 20 data: %.4f" % err20)

20 data: [['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'tidak', 'ada'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'ada', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'tidak', 'tidak'], ['panjang', 'ada', 'ada', 'ada'], ['panjang', 'ada', 'ada', 'ada'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'ada', 'ada'], ['panjang', 'ada', 'ada', 'ada'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'tidak', 'ada', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['pendek', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'tidak', 'tidak'], ['panjang', 'tidak', 'tidak', 'tidak'], ['panjang', 'ada', 'tidak', 'tidak']]
label  : ['ham', 'ham', 'spam', 'ham', 'ham', 'spam', 'ham', 'ham', 'spam', 'spam', 'ham', 'spam', 'spam', 'ham', 'ham', 'spam', 'ham', 'ham', 'ham', 'spam']

prior: {'ham': 0.6, 'spam': 0.4}
jumlah kelas: {'ham': 12, 'spam': 8}

l

In [13]:
# soal 4: bandingkan dengan Decision Tree pada 20 data yang sama
# fungsi DT (entropy/gain/buat_tree) dibawa dari notebook desiciontree
import math

def entropy(data):
    n = len(data)
    ent = 0.0
    for c in Counter(data).values():
        p = c / n
        ent -= p * math.log2(p)
    return ent

def gain(data_input, data_output, fitur_idx):
    base_ent = entropy(data_output)
    base_n = len(data_output)
    subsets = {}
    for x, label in zip(data_input, data_output):
        subsets.setdefault(x[fitur_idx], []).append(label)
    subset_ent = sum((len(v) / base_n) * entropy(v) for v in subsets.values())
    return base_ent - subset_ent

def buat_tree(data_input, data_output, feature_indices):
    if len(set(data_output)) == 1:
        return data_output[0]
    if not feature_indices:
        return Counter(data_output).most_common(1)[0][0]
    gains = sorted([(gain(data_input, data_output, i), i) for i in feature_indices], reverse=True)
    best_gain, best_fitur = gains[0]
    if best_gain == 0:
        return Counter(data_output).most_common(1)[0][0]
    tree = {best_fitur: {}}
    for val in set(x[best_fitur] for x in data_input):
        sub_in = [x for x, l in zip(data_input, data_output) if x[best_fitur] == val]
        sub_out = [l for x, l in zip(data_input, data_output) if x[best_fitur] == val]
        sisa = [i for i in feature_indices if i != best_fitur]
        tree[best_fitur][val] = buat_tree(sub_in, sub_out, sisa) if sub_in else Counter(data_output).most_common(1)[0][0]
    return tree

def prediksi_tree(tree, data):
    if not isinstance(tree, dict):
        return tree
    idx = next(iter(tree))
    cabang = tree[idx]
    if data[idx] in cabang:
        return prediksi_tree(cabang[data[idx]], data)
    daun = [v for v in cabang.values() if not isinstance(v, dict)]
    return Counter(daun).most_common(1)[0][0] if daun else prediksi_tree(next(iter(cabang.values())), data)

# jalankan DT pada x20/y20 yang sama (fitur sudah berupa string kategorikal)
tree20_dt = buat_tree(x20, y20, list(range(len(fitur))))
y_pred_dt20 = [prediksi_tree(tree20_dt, xi) for xi in x20]

acc_dt20 = akurasi(y20, y_pred_dt20)
y_pred_dt20_num = [label_angka20[v] for v in y_pred_dt20]
err_dt20 = rmse(y20_num, y_pred_dt20_num)

print("decision tree 20 data:", tree20_dt)
print("prediksi DT:", y_pred_dt20)
print()
print("== Perbandingan 20 data ==")
print("Naive Bayes : akurasi=%.4f (%.2f%%)  RMSE=%.4f" % (acc20, acc20 * 100, err20))
print("Decision Tree: akurasi=%.4f (%.2f%%)  RMSE=%.4f" % (acc_dt20, acc_dt20 * 100, err_dt20))
print("data salah NB :", [i + 1 for i in range(len(y20)) if y20[i] != y_pred20[i]])
print("data salah DT :", [i + 1 for i in range(len(y20)) if y20[i] != y_pred_dt20[i]])

decision tree 20 data: {3: {'tidak': {1: {'tidak': {2: {'tidak': {0: {'pendek': 'ham', 'panjang': 'ham'}}, 'ada': 'ham'}}, 'ada': {2: {'tidak': 'ham', 'ada': 'spam'}}}}, 'ada': 'spam'}}
prediksi DT: ['ham', 'ham', 'spam', 'ham', 'ham', 'spam', 'ham', 'ham', 'spam', 'spam', 'ham', 'spam', 'spam', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham', 'ham']

== Perbandingan 20 data ==
Naive Bayes : akurasi=0.9000 (90.00%)  RMSE=0.3162
Decision Tree: akurasi=0.9000 (90.00%)  RMSE=0.3162
data salah NB : [16, 20]
data salah DT : [16, 20]
